# Bayesian inference on the 2D Ising model — a guided walkthrough

This notebook is the **guided, intermediate-level** companion to the lab-style README. It runs the same pipeline as `python main.py` (simulate → infer → diagnose), but it stops at every decision point to explain *why* the choice is made — the likelihood, the priors, the sampler, and how to read the convergence diagnostics. For the full derivations see the README; for a bare end-to-end run see `main.py`.

**What you will do, end to end:**
1. look at the two phases of the model directly,
2. simulate the magnetization curve `M(T)` that serves as data,
3. build a Bayesian model for the critical parameters `(Tc, β)`,
4. sample the posterior with Metropolis-Hastings,
5. check that the sampler converged, and
6. compare against Onsager's 1944 exact solution.

**Run order & artifacts.** Run cells top to bottom from the project root. Section 3 writes `data/magnetization.csv`, Section 5 writes `data/trace.nc`, and Section 9 writes fresh PNGs to `results/` (a gitignored staging folder), so the tracked `figures/` are left untouched. Because the Numba RNG is not reproducible across runs, your numbers will differ from the frozen README values by a small Monte-Carlo margin — the *conclusions* (β ≈ 1/8, `Tc` shifted upward on a finite lattice) are what should reproduce, not the last digit.

## 1 · Setup

We pin Onsager's exact results at the top so every later cell can be checked against them on the spot. These are the *ground truth* for an infinite lattice: `Tc = 2/ln(1+√2)` and `β = 1/8`. Everything the pipeline produces is an estimate of these two numbers from finite, noisy simulation.

In [ ]:
import os
# Run from the project root so the relative data/ and results/ paths resolve.
if os.getcwd().endswith('notebooks'):
    os.chdir('..')
print('working directory:', os.getcwd())

import numpy as np
import matplotlib.pyplot as plt

from src import metropolis, bayesian, plots

# Onsager (1944), exact for the infinite 2D lattice — our yardstick throughout.
ONSAGER_TC = 2.0 / np.log(1.0 + np.sqrt(2.0))
EXACT_BETA = 0.125
print(f'Onsager exact:  Tc = {ONSAGER_TC:.4f}   β = {EXACT_BETA}')

# Portfolio palette, reused so the inline plots match the README figures.
PURPLE, CYAN, PINK, AMBER = '#7c5cff', '#22d3ee', '#f472b6', '#fbbf24'

## 2 · A picture of the two phases

Before measuring anything, look at what the system *is* on each side of the transition. We equilibrate two independent 32×32 lattices — one cold, one hot — and view them as images. Below `Tc` the spins align into one of two ground states (the model's `Z₂` symmetry: flipping every spin costs nothing); above `Tc` thermal noise wins and the magnetization averages to zero. No fit is needed to see that *something* qualitative happens in between — that qualitative change is the whole subject of the notebook.

Note these two snapshots start from a **random** configuration: for a single illustrative picture that is fine. The production sweep in the next section starts from an **ordered** lattice instead, for a reason we explain there.

In [ ]:
def run_to_equilibrium(T, size=32, n_sweeps=2000, seed=0):
    """Relax one lattice to equilibrium at temperature T and return it."""
    np.random.seed(seed)
    lat = np.where(np.random.random((size, size)) < 0.5, -1, 1).astype(np.int8)
    for _ in range(n_sweeps):
        metropolis._sweep(lat, T, 1.0)  # one sweep = N² attempted spin flips
    return lat

lat_cold = run_to_equilibrium(T=1.8, seed=0)   # well below Tc
lat_hot  = run_to_equilibrium(T=3.2, seed=1)   # well above Tc

fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.6))
titles = [
    fr'$T = 1.8$  $(T < T_c)$ — ordered, $\langle |M| \rangle \approx {np.abs(lat_cold.mean()):.2f}$',
    fr'$T = 3.2$  $(T > T_c)$ — disordered, $\langle |M| \rangle \approx {np.abs(lat_hot.mean()):.2f}$',
]
for ax, lat, title in zip(axes, [lat_cold, lat_hot], titles):
    ax.imshow(lat, cmap='PuOr', vmin=-1, vmax=1, interpolation='nearest')
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('One equilibrium configuration in each phase  ·  32×32 lattice', fontweight='bold')
fig.tight_layout()
plt.show()

## 3 · The temperature sweep — building the "data"

This is the expensive step: we scan 25 temperatures from `T = 1.5` to `3.5` with the exact settings `main.py` uses, so the output matches `data/magnetization.csv`. For each temperature we equilibrate, then average `⟨|M|⟩` over many sweeps. Runtime ≈ 30–90 s with Numba.

**Why an ordered (cold) start.** `simulate_at_temperature` initializes every spin to `+1` rather than randomly. A random start can freeze the low-`T` runs into a long-lived *domain-wall* (stripe) state — metastable and exponentially slow to anneal out — which biases `⟨|M|⟩` downward 10–30 % of the time. Starting from the ordered ground state samples the broken-symmetry phase cleanly; above `Tc` the burn-in disorders it anyway, so nothing is lost.

**Why the error bars swell near `Tc`.** Close to the transition the correlation time diverges (*critical slowing down*), so successive sweeps are highly correlated and the effective sample size drops — visible as the largest `σ_M` right around the transition.

In [ ]:
results = metropolis.run_full_simulation(
    out_csv='data/magnetization.csv',
    size=32, n_temps=25, t_min=1.5, t_max=3.5,
    n_therm=2000, n_measure=3000, seed=0,   # same knobs as main.py → comparable output
)

T  = np.array([r['T']      for r in results])
M  = np.array([r['M_mean'] for r in results])
Sd = np.array([r['M_std']  for r in results])

fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.axvspan(T.min(), ONSAGER_TC, color=PURPLE, alpha=0.06)   # ordered side
ax.axvspan(ONSAGER_TC, T.max(), color=AMBER, alpha=0.06)    # disordered side
ax.errorbar(T, M, yerr=Sd, fmt='o', color=PURPLE,
            markeredgecolor='white', ecolor='#6b7280', capsize=2.5,
            label=r'$\langle |M| \rangle \pm \sigma_M$')
ax.axvline(ONSAGER_TC, color=PINK, ls='--', lw=1.8, label='Onsager $T_c$')
ax.set_xlabel(r'Temperature  $T$   ($J/k_B$)')
ax.set_ylabel(r'Magnetization per site  $\langle |M| \rangle$')
ax.set_title('Simulated M(T) — the data for the Bayesian stage', loc='left')
ax.grid(alpha=0.3); ax.legend(loc='lower left')
plt.tight_layout(); plt.show()

## 4 · Building the Bayesian model

Now we turn the curve into numbers. Statistical mechanics says that just below a second-order transition the order parameter vanishes as a power law of the reduced temperature:

$$\langle |M| \rangle(T) \;\approx\; (1 - T/T_c)^{\beta} \quad (T < T_c), \qquad \langle |M| \rangle(T) = 0 \quad (T > T_c).$$

So the *shape* of the curve carries two unknowns: the critical temperature `Tc` and the critical exponent `β` (theory: `β = 1/8` exactly for this universality class). We treat the simulated points as noisy observations of that piecewise mean and infer `(Tc, β)` — plus a noise scale `σ` — with Bayes' theorem.

**Priors — what we assert before seeing the data.** Each is deliberately *loose* so the likelihood dominates, but informative enough to keep the sampler in physically sensible territory:

| Parameter | Prior | Why this choice |
|---|---|---|
| `Tc` | `Normal(2.3, 0.3)` | broad envelope around published `Tc` for small 2D lattices; σ=0.3 spans well past the finite-size shift |
| `β`  | `Normal(0.12, 0.05)` | centered near the 2D Ising exponent without hard-coding `1/8`; lets the data pull it |
| `σ`  | `HalfNormal(0.1)` | positive-only scale for the unexplained scatter in `⟨|M|⟩` |

**Likelihood — Gaussian noise around the piecewise mean:** `M_obs ~ Normal(M_pred(Tc, β), σ)`, with `M_pred = 0` for `T > Tc` (the broken-symmetry argument). The `T > Tc` branch is what lets the model *locate* `Tc`: it is the temperature where the power-law arm switches off.

The log-posterior `log P(data|θ) + log P(θ)` is implemented directly in `src/bayesian.py`; there is no closed form, so we sample it. The whole model is three short functions — worth reading once:

In [ ]:
import inspect
print(inspect.getsource(bayesian._log_prior))
print(inspect.getsource(bayesian._log_likelihood))

## 5 · Sampling the posterior

We use the **random-walk Metropolis-Hastings** sampler (`backend='numpy'`): it only needs to *evaluate* the log-posterior — no gradients, no C compiler — which makes the notebook run anywhere. (A PyMC + NUTS backend is available with `backend='pymc'` and reaches the same effective sample size with far fewer draws if you have a working PyTensor.)

The knobs: **4 chains** started from different points (so disagreement between them flags non-convergence), **1000 tune** steps discarded as burn-in, **5000 draws** kept per chain. The acceptance rate lands in the low tens of percent — the efficient regime for random-walk proposals, which is optimal near `≈23 %` in several dimensions. Much higher would mean the steps are too small (slow exploration); much lower means most proposals are rejected. The printout below reports it per chain.

In [ ]:
trace = bayesian.run_full_inference(
    csv_in='data/magnetization.csv',
    trace_out='data/trace.nc',
    backend='numpy',
    draws=5000, tune=1000, chains=4, seed=0,
)

In [ ]:
import arviz as az
# Posterior summary: mean, sd, 95% HDI, plus R̂ and ESS per parameter.
az.summary(trace, var_names=['Tc', 'beta', 'sigma'], hdi_prob=0.95)

## 6 · Reading the posteriors

The numbers above are marginal posteriors — full distributions, not point estimates. Read each as *mean ± sd* with a 95 % highest-density interval (HDI), the narrowest interval holding 95 % of the probability mass.

Two things to look for:
- **`β` should bracket `1/8 = 0.125`.** If the exact exponent sits inside its 95 % HDI, the piecewise power law is the right functional form and the data constrain it.
- **`Tc` will land slightly *above* Onsager's `2.2692`.** That is not an error — it is the finite-size shift, quantified in Section 8.

Let us plot the two marginals with their HDIs and the exact values.

In [ ]:
tc = trace.posterior['Tc'].values.ravel()
bt = trace.posterior['beta'].values.ravel()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, s, exact, color, name in [
    (axes[0], tc, ONSAGER_TC, PURPLE, r'$T_c$'),
    (axes[1], bt, EXACT_BETA, CYAN,   r'$\beta$'),
]:
    lo, hi = az.hdi(s, hdi_prob=0.95)
    ax.hist(s, bins=60, density=True, color=color, alpha=0.45)
    ax.axvline(s.mean(), color=PINK, lw=2, label=f'posterior mean {s.mean():.3f}')
    ax.axvline(exact, color=AMBER, ls='--', lw=2, label=f'exact {exact:.4f}')
    ax.axvspan(lo, hi, color=color, alpha=0.12, label=f'95% HDI [{lo:.3f}, {hi:.3f}]')
    ax.set_title(f'Posterior of {name}', loc='left')
    ax.legend(fontsize=8, loc='upper right')
fig.tight_layout(); plt.show()

## 7 · MCMC diagnostics — did the sampler actually work?

A sampler that has not converged returns numbers that *look* like samples but are not. Three non-negotiable checks:
- **`R̂` (Gelman-Rubin)** compares between-chain to within-chain variance. `≤ 1.01` is ideal, `≤ 1.05` tolerable.
- **`ESS_bulk`** is the effective sample size after discounting autocorrelation; `≥ 200` is a working floor.
- **Visual trace inspection**: healthy chains overlap and look like stationary noise, not separated bands or slow drifts.

One honest subtlety here: `β` is the **slowest-mixing** parameter. The hard threshold at `T = Tc` in the likelihood makes its posterior strongly autocorrelated, so its `ESS_bulk` sits near the 200 floor and can dip below it on a given run while `R̂` still passes. That is a property of random-walk MH on this likelihood, not a broken model — more draws or the gradient-based NUTS backend lift it comfortably.

In [ ]:
rhat = {v: float(az.rhat(trace)[v]) for v in ['Tc', 'beta', 'sigma']}
ess  = {v: float(az.ess(trace)[v])  for v in ['Tc', 'beta', 'sigma']}
print('R_hat   :', {k: round(v, 3) for k, v in rhat.items()})
print('ESS_bulk:', {k: int(v)      for k, v in ess.items()})

# R̂ should pass for all three. β often sits right at the ESS floor (see note above).
print('\nR̂ ≤ 1.05 for all?      ', all(v <= 1.05 for v in rhat.values()))
print('ESS_bulk ≥ 200 for all? ', all(v >= 200 for v in ess.values()),
      '  (β is the borderline one — slowest-mixing parameter)')

# Per-chain trace: look for four interleaved bands, no drift.
fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
for ax, key, exact in zip(axes, ['Tc', 'beta', 'sigma'], [ONSAGER_TC, EXACT_BETA, None]):
    for chain in trace.posterior[key].values:
        ax.plot(chain, lw=0.5, alpha=0.7)
    if exact is not None:
        ax.axhline(exact, color=AMBER, ls='--', lw=1.5)
    ax.set_ylabel(key)
axes[-1].set_xlabel('MCMC iteration (post burn-in)')
fig.suptitle('Per-chain traces — should interleave like noise', fontweight='bold')
fig.tight_layout(); plt.show()

## 8 · Ground-truth check against Onsager (1944)

Finally, hold the inference up against the exact solution. The validation rules follow directly from theory:
- the 95 % HDI of `β` **should contain** `1/8` — the exponent is *universal*, independent of lattice size;
- the 95 % HDI of `Tc` is **expected to sit above** Onsager's `2.2692` — on a finite `32×32` lattice the correlation length hits the boundaries and shifts the apparent transition up by `~+0.1` (Ferrenberg & Landau 1991), shrinking as `1/N`.

So a `β` that brackets `1/8` *and* a `Tc` shifted upward is the model behaving correctly — the shift is physics, not a sampler bug. (Enlarge the lattice to watch it shrink; see README §13.)

In [ ]:
beta_hdi = az.hdi(bt, hdi_prob=0.95)
tc_hdi   = az.hdi(tc, hdi_prob=0.95)

print(f'β : 95% HDI = [{beta_hdi[0]:.3f}, {beta_hdi[1]:.3f}]   '
      f'contains 1/8 = {EXACT_BETA}?  {beta_hdi[0] <= EXACT_BETA <= beta_hdi[1]}')
print(f'Tc: 95% HDI = [{tc_hdi[0]:.3f}, {tc_hdi[1]:.3f}]   '
      f'posterior mean {tc.mean():.3f} vs Onsager {ONSAGER_TC:.4f}  '
      f'(+{tc.mean() - ONSAGER_TC:.3f}, finite-size shift expected)')

## 9 · (Optional) Regenerate the README figures

`src.plots.plot_all` is exactly what runs at the end of `python main.py`. Here we point it at `results/` — a gitignored staging folder — so re-running the notebook never clobbers the tracked PNGs in `figures/`.

In [ ]:
plots.plot_all(
    csv_path='data/magnetization.csv',
    trace_path='data/trace.nc',
    out_dir='results',      # staging dir; tracked figures/ left untouched
    lattice_size=32,
)
print('fresh PNGs written to results/')